In [2]:
import sys
import os
sys.path.append(os.getcwd())

In [1]:
import json
import os
from docling.document_converter import DocumentConverter, PdfFormatOption
from docling.datamodel.base_models import InputFormat
from docling.datamodel.pipeline_options import PdfPipelineOptions, TableFormerMode
from docling.chunking import HierarchicalChunker

c:\Users\Dell\anaconda3\envs\slm_rag\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [6]:
pdf_file = "../data/pdfs/Liasi - Règlement d'application - 19-06-2007 - 31-12-2024.pdf"

In [4]:
pipeline_options = PdfPipelineOptions(
    do_ocr=True,                      
    do_table_structure=True,          
    table_structure_options={"mode": TableFormerMode.ACCURATE} 
)

converter = DocumentConverter(
    format_options={
        InputFormat.PDF: PdfFormatOption(pipeline_options=pipeline_options)
    }
)

In [7]:
result = converter.convert(pdf_file)
doc = result.document

2026-01-11 00:26:06,957 - INFO - detected formats: [<InputFormat.PDF: 'pdf'>]
2026-01-11 00:26:07,080 - INFO - Going to convert document batch...
2026-01-11 00:26:07,081 - INFO - Initializing pipeline for StandardPdfPipeline with options hash 5a43216f093a7c32c3d8090bbb471faa
2026-01-11 00:26:07,111 - INFO - Loading plugin 'docling_defaults'
2026-01-11 00:26:07,116 - INFO - Registered ocr engines: ['easyocr', 'ocrmac', 'rapidocr', 'tesserocr', 'tesseract']
2026-01-11 00:26:08,643 - INFO - Accelerator device: 'cuda:0'
2026-01-11 00:26:10,088 - INFO - Accelerator device: 'cuda:0'
2026-01-11 00:26:11,555 - INFO - Accelerator device: 'cuda:0'
2026-01-11 00:26:12,032 - INFO - Loading plugin 'docling_defaults'
2026-01-11 00:26:12,032 - INFO - Registered picture descriptions: ['vlm', 'api']
2026-01-11 00:26:12,032 - INFO - Processing document Liasi - Règlement d'application - 19-06-2007 - 31-12-2024.pdf
2026-01-11 00:26:22,351 - INFO - Finished converting document Liasi - Règlement d'applicati

In [14]:
# 1. INSPECT THE RESULT WRAPPER
print(f"📦 Type: {type(result)}")
print(f"✅ Status: {result.status}")  # Should be 'ConversionStatus.SUCCESS'
# print(f"⏱️ Time Taken: {result.timings['pipeline_conversion']} seconds")
print(f"❌ Errors: {result.errors}")   # Should be empty []

📦 Type: <class 'docling.datamodel.document.ConversionResult'>
✅ Status: ConversionStatus.SUCCESS
❌ Errors: []


In [17]:
print(f"Pages Found: {len(doc.pages)}")
print(f"Tables Found: {len(doc.tables)}")
print(f"Images Found: {len(doc.pictures)}")
print(f"Total Text Items: {len(doc.texts)}")

Pages Found: 19
Tables Found: 2
Images Found: 0
Total Text Items: 551


In [20]:
print("\n--- 🧠 FIRST 5 ITEMS DETECTED ---")
# doc.texts is a flat list of every text element
for i, item in enumerate(doc.texts[:15]): 
    label = item.label.value # e.g., 'text', 'section_header', 'table'
    text = item.text.strip()
    page = item.prov[0].page_no if item.prov else "N/A"
    
    print(f"[{i}] Page {page} | Type: {label} | Content: '{text[:50]}...'")


--- 🧠 FIRST 5 ITEMS DETECTED ---
[0] Page 1 | Type: page_header | Content: 'rsGE J 4 04.01: Règlement d'exécution de la loi su...'
[1] Page 1 | Type: section_header | Content: 'Source SILGENEVE PUBLIC...'
[2] Page 1 | Type: section_header | Content: 'Dernières modifications au 1 er  juin 2024...'
[3] Page 1 | Type: text | Content: 'Règlement d'exécution de la loi sur l'insertion et...'
[4] Page 1 | Type: text | Content: 'J 4 04.01...'
[5] Page 1 | Type: text | Content: 'du 25 juillet 2007...'
[6] Page 1 | Type: text | Content: '(Entrée en vigueur : 1 er  août 2007)...'
[7] Page 1 | Type: text | Content: 'Le CONSEIL D'ÉTAT de la République et canton de Ge...'
[8] Page 1 | Type: text | Content: 'vu la loi sur l'insertion et l'aide sociale indivi...'
[9] Page 1 | Type: text | Content: 'vu l'ordonnance 11 concernant les adaptations dans...'
[10] Page 1 | Type: text | Content: 'arrête :...'
[11] Page 1 | Type: section_header | Content: 'Chapitre I        Conditions et mode de calcul des...

In [21]:
chunker = HierarchicalChunker(
    merge_list_items=True
)

raw_chunks = []

In [30]:
for chunk in chunker.chunk(doc):
    print(chunk)
    page_number = None
    if hasattr(chunk.meta, 'doc_items'):
        for item in chunk.meta.doc_items:
            # print(item)
            if hasattr(item, 'prov') and item.prov:
                # print(item.prov)
                continue

text="Règlement d'exécution de la loi sur l'insertion et l'aide sociale individuelle (7) (RIASI)" meta=DocMeta(schema_name='docling_core.transforms.chunker.DocMeta', version='1.0.0', doc_items=[TextItem(self_ref='#/texts/3', parent=RefItem(cref='#/groups/1'), children=[], content_layer=<ContentLayer.BODY: 'body'>, meta=None, label=<DocItemLabel.TEXT: 'text'>, prov=[ProvenanceItem(page_no=1, bbox=BoundingBox(l=57.36, t=711.0390439453125, r=254.847, b=654.5410439453126, coord_origin=<CoordOrigin.BOTTOMLEFT: 'BOTTOMLEFT'>), charspan=(0, 90))], orig="Règlement d'exécution de la loi sur l'insertion et l'aide sociale individuelle (7) (RIASI)", text="Règlement d'exécution de la loi sur l'insertion et l'aide sociale individuelle (7) (RIASI)", formatting=None, hyperlink=None)], headings=['Dernières modifications au 1 er  juin 2024'], captions=None, origin=DocumentOrigin(mimetype='application/pdf', binary_hash=16623852833566215495, filename="Liasi - Règlement d'application - 19-06-2007 - 31-12-2